# Phase 6: Waiting Time & No-Show Predictions
This notebook trains and evaluates a Random Forest Regressor to predict customer waiting time in minutes, and a Logistic Regression classifier to predict the no-show probability of appointments.

> **Superseded for serving.** This notebook is the original exploration. The models the API serves are trained by `scripts/train_models.py`, which fixes the historical-average leakage, uses time-based splits, and writes measured metrics to `backend/app/ml_models/model_card.json`. Use those metrics, not the outputs below.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, classification_report
import pickle
import os

print("Libraries loaded.")

Libraries loaded.


## 1. Load Data

In [2]:
visits = pd.read_csv('../data/processed/visits.csv')
shops = pd.read_csv('../data/processed/shops.csv')
appointments = pd.read_csv('../data/processed/appointments.csv')
customers = pd.read_csv('../data/processed/customers.csv')

## 2. Prepare Data for Waiting Time Prediction
We merge features such as queue length, occupied seats, and shop capacity from availability and visits tables.

In [3]:
# In visits: we have arrival_time (HH:MM) and date
# We will match the closest availability log at that date & hour to capture queue state.
visits['hour'] = pd.to_datetime(visits['arrival_time']).dt.hour
visits['day_of_week'] = pd.to_datetime(visits['date']).dt.dayofweek

# Clean availability timestamps to extract hour and date
availability = pd.read_csv('../data/processed/availability.csv')
availability['timestamp_dt'] = pd.to_datetime(availability['timestamp'])
availability['date_str'] = availability['timestamp_dt'].dt.strftime('%Y-%m-%d')
availability['hour'] = availability['timestamp_dt'].dt.hour

# Join visits with availability on (shop_id, date, hour)
# To avoid double joins, let's keep only unique availability rows
avail_unique = availability.drop_duplicates(subset=['shop_id', 'date_str', 'hour'])

wt_df = visits.merge(avail_unique[['shop_id', 'date_str', 'hour', 'queue_length', 'occupied_seats', 'available_seats']], 
                     left_on=['shop_id', 'date', 'hour'], 
                     right_on=['shop_id', 'date_str', 'hour'], 
                     how='left')

# Drop null joins (hours outside standard snapshots) and fillna
wt_df['queue_length'] = wt_df['queue_length'].fillna(0)
wt_df['occupied_seats'] = wt_df['occupied_seats'].fillna(0)
wt_df['available_seats'] = wt_df['available_seats'].fillna(wt_df['seat_id'])

# Add shop metrics
wt_df = wt_df.merge(shops[['shop_id', 'number_of_barbers', 'number_of_seats']], on='shop_id')

wt_cols = ['queue_length', 'occupied_seats', 'available_seats', 'number_of_barbers', 'number_of_seats', 'hour', 'day_of_week']
X_wt = wt_df[wt_cols]
y_wt = wt_df['waiting_time_minutes']

print(f"Features: {wt_cols}")
print(f"Total rows: {len(X_wt)}")

Features: ['queue_length', 'occupied_seats', 'available_seats', 'number_of_barbers', 'number_of_seats', 'hour', 'day_of_week']
Total rows: 64991


C:\Users\PC\AppData\Local\Temp\ipykernel_18776\1572875464.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  visits['hour'] = pd.to_datetime(visits['arrival_time']).dt.hour


## 3. Train Waiting Time Model (Random Forest)

In [4]:
X_train_wt, X_test_wt, y_train_wt, y_test_wt = train_test_split(X_wt, y_wt, test_size=0.2, random_state=42)

rf_wt = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_wt.fit(X_train_wt, y_train_wt)
wt_preds = rf_wt.predict(X_test_wt)

print("Waiting Time Predictor Metrics:")
print(f" - MAE: {mean_absolute_error(y_test_wt, wt_preds):.3f} minutes")
print(f" - RMSE: {np.sqrt(mean_squared_error(y_test_wt, wt_preds)):.3f}")
print(f" - R2: {r2_score(y_test_wt, wt_preds):.3f}")

# Save wait time model
with open('../models/waiting_time_model.pkl', 'wb') as f:
    pickle.dump(rf_wt, f)
print("Saved waiting_time_model.pkl successfully.")

Waiting Time Predictor Metrics:
 - MAE: 8.079 minutes
 - RMSE: 19.704
 - R2: 0.871
Saved waiting_time_model.pkl successfully.


## 4. Appointment No-Show Prediction (Classification)
Goal: Predict `no_show_flag` utilizing Logistic Regression.

In [5]:
# Merge customer demographics and appointment parameters
app_df = appointments.merge(customers[['customer_id', 'age_group', 'gender', 'customer_rating']], on='customer_id')

# Encode categorical values
app_df['gender_Male'] = (app_df['gender'] == 'Male').astype(int)
app_df['gender_Female'] = (app_df['gender'] == 'Female').astype(int)

# Encode age groups
for age in ['18-25', '26-35', '36-45', '46-60', '60+']:
    app_df[f'age_{age}'] = (app_df['age_group'] == age).astype(int)

# Extract hour of appointment
app_df['hour'] = pd.to_datetime(app_df['appointment_time']).dt.hour

# Calculate booking lag (days between booking and appointment)
app_df['booking_dt'] = pd.to_datetime(app_df['booking_time'])
app_df['app_dt'] = pd.to_datetime(app_df['appointment_date'] + ' ' + app_df['appointment_time'])
app_df['booking_lag_days'] = (app_df['app_dt'] - app_df['booking_dt']).dt.total_seconds() / (24 * 3600)

no_show_cols = ['gender_Male', 'gender_Female', 'age_18-25', 'age_26-35', 'age_36-45', 'age_46-60', 'age_60+', 'hour', 'customer_rating', 'booking_lag_days']
X_ns = app_df[no_show_cols]
y_ns = app_df['no_show_flag']

print(f"No-show features: {no_show_cols}")
print(f"Target count: {y_ns.value_counts()}")

No-show features: ['gender_Male', 'gender_Female', 'age_18-25', 'age_26-35', 'age_36-45', 'age_46-60', 'age_60+', 'hour', 'customer_rating', 'booking_lag_days']
Target count: no_show_flag
0    30210
1     2312
Name: count, dtype: int64


C:\Users\PC\AppData\Local\Temp\ipykernel_18776\1604089977.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  app_df['hour'] = pd.to_datetime(app_df['appointment_time']).dt.hour


## 5. Train No-Show Classifier

In [6]:
X_train_ns, X_test_ns, y_train_ns, y_test_ns = train_test_split(X_ns, y_ns, test_size=0.2, random_state=42, stratify=y_ns)

lr_ns = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_ns.fit(X_train_ns, y_train_ns)
ns_preds = lr_ns.predict(X_test_ns)

print("No-Show Classifier Performance:")
print(f"Accuracy: {accuracy_score(y_test_ns, ns_preds):.3f}")
print("\nClassification Report:")
print(classification_report(y_test_ns, ns_preds))

with open('../models/noshow_model.pkl', 'wb') as f:
    pickle.dump(lr_ns, f)
print("Saved noshow_model.pkl successfully.")

No-Show Classifier Performance:
Accuracy: 0.672

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.67      0.79      6043
           1       0.13      0.63      0.21       462

    accuracy                           0.67      6505
   macro avg       0.54      0.65      0.50      6505
weighted avg       0.90      0.67      0.75      6505

Saved noshow_model.pkl successfully.
